# Cleaning_CYRV_order_payments_dataset.csv

## Data Cleaning Summary

### Conclusion

The dataset was inspected for missing values, duplicate records, invalid categories, unusual installment values, zero payment values, and potential outliers.

No records were removed or modified because the identified unusual values could not be confirmed as data errors. The original data was therefore retained.

### Data Cleaning Summary

| Check | Finding | Decision |
|---|---|---|
| Missing values | No missing values were found | No action required |
| Duplicate rows | No duplicate rows were found | No action required |
| Data types | Data types are appropriate for all columns | No action required |
| `payment_type` | 3 records with `not_defined` | Keep — the actual payment type cannot be determined |
| `payment_installments` | 2 records with `0` installments | Keep — the correct value cannot be determined |
| `payment_value` | Several records with a value of `0` | Keep — zero values may be valid |
| High `payment_installments` | Maximum value is 24 | Keep — high values are associated with credit card payments |
| High `payment_sequential` | Maximum value is 29 | Keep — multiple voucher payments exist for the same order |
| High `payment_value` | Maximum value is 13,664.08 | Keep — high values are possible outliers, but not necessarily errors |


## 1. Inspection

In [5]:
import pandas as pd

payments = pd.read_csv("/Users/yuliiapotrymai/Desktop/SpikupCapstone2026_CYRV/data/cyrv/CYRV_order_payments_dataset.csv")

payments.head()


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [6]:
print("Dataset shape:", payments.shape)

print("\nColumns:")
print(payments.columns.tolist())

print("\nData types:")
print(payments.dtypes)


Dataset shape: (103886, 5)

Columns:
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Data types:
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object


## 2. Validation

In [35]:
payments.isna().sum() #missing values

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [8]:
missing = pd.DataFrame({
    "missing_count": payments.isna().sum(),
    "missing_percent": payments.isna().mean() * 100
})

missing.sort_values("missing_count", ascending=False)

,missing_count,missing_percent
order_id,0,0.0
payment_sequential,0,0.0
payment_type,0,0.0
payment_installments,0,0.0
payment_value,0,0.0


In [9]:
print("Duplicate rows:", payments.duplicated().sum())

Duplicate rows: 0


In [10]:
payments.describe()

,payment_sequential,payment_installments,payment_value
count,103886.000000,103886.000000,103886.000000
mean,1.092679,2.853349,154.100380
std,0.706584,2.687051,217.494064
min,1.000000,0.000000,0.000000
25%,1.000000,1.000000,56.790000
50%,1.000000,1.000000,100.000000
75%,1.000000,4.000000,171.837500
max,29.000000,24.000000,13664.080000


In [14]:
payments.describe(include="str")


,order_id,payment_type
count,103886,103886
unique,99440,5
top,fa65dad1b0e818e3ccc5cb0e39231352,credit_card
freq,29,76795


In [15]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [17]:
payments["payment_installments"].value_counts().sort_index()

payment_installments
0         2
1     52546
2     12413
3     10461
4      7098
5      5239
6      3920
7      1626
8      4268
9       644
10     5328
11       23
12      133
13       16
14       15
15       74
16        5
17        8
18       27
20       17
21        3
22        1
23        1
24       18
Name: count, dtype: int64

In [18]:
payments["payment_value"].sort_values().head(20)

62674     0.00
36822     0.00
100766    0.00
94427     0.00
77885     0.00
51280     0.00
57411     0.00
19922     0.00
43744     0.00
810       0.01
17931     0.01
64471     0.01
55432     0.01
2522      0.01
64178     0.01
89860     0.03
11395     0.03
56841     0.05
49555     0.05
26330     0.07
Name: payment_value, dtype: float64

Issues at this stage
Issue 1: payment_type = "not_defined" - 3 records\
Issue 2: payment_installments = 0 - 2 records\
Issue 3: payment_value = 0 -  several records\
Issue 4: Some payment records have a very large number of installments: 20–24


## 3. Investigation

In [24]:
payments[
    (payments["payment_type"] == "not_defined") |
    (payments["payment_installments"] == 0) |
    (payments["payment_value"] == 0)
].sort_values(
    ["order_id", "payment_sequential"]
)


,order_id,payment_sequential,payment_type,payment_installments,payment_value
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.00
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.00
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.00
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.00
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.00
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.00
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.00
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.00


### We have 11 rows that met our three conditions:
payment_type == "not_defined"
payment_installments == 0
payment_value == 0

1. payment_value = 0 + payment_type = voucher . A voucher is not necessarily a standalone monetary payment. A voucher can fully cover a portion of the order cost, so a separate payment record may have a `payment_value` of 0.
Therefore:
We will not automatically treat `payment_value = 0` as an error.
2. payment_installments = 0 - It doesn't make sense, looks like a potential data quality issue.
3. not_defined. All three share the same combination: "not_defined + 1 installment + 0.00". This may be a special value from the source dataset indicating that the payment method was not defined.
4. fa65dad1b0e818e3ccc5cb0e39231352	 - has payment_sequential = 14 and 13; payment_type = voucher; payment_value = 0.00

### Decision at this stage
payment_value = 0 + voucher	6	Keep  
payment_type = not_defined	3	Investigate / keep for now  
payment_installments = 0	2	Investigate  
payment_value = 0  	6	Don't delete  


In [25]:
payments[payments["payment_type"] == "not_defined"]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [26]:
not_defined_orders = payments.loc[
    payments["payment_type"] == "not_defined",
    "order_id"
]

payments[payments["order_id"].isin(not_defined_orders)].sort_values(
    ["order_id", "payment_sequential"]
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


Keep not_defined values because the actual payment type cannot be determined from the available payment data.

In [27]:
zero_installment_orders = payments.loc[
    payments["payment_installments"] == 0,
    "order_id"
]

payments[
    payments["order_id"].isin(zero_installment_orders)
].sort_values(
    ["order_id", "payment_sequential"]
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


In [30]:
payments[
    payments["order_id"].isin([
        "1a57108394169c0b47d8f876acc9ba2d",
        "744bade1fcf9ff3f31d860ace076d422"
    ])
].sort_values(["order_id", "payment_sequential"])


,order_id,payment_sequential,payment_type,payment_installments,payment_value
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


Keep the records and flag payment_installments = 0 as an anomalous value because the correct number of installments cannot be determined from the available data.
To cross-check these `order_id`s between the tables when working with `CYRV_orders_dataset.csv` and `CYRV_order_items_dataset.csv`.

### Checking for extreme values

In [31]:
payments.nlargest(10, "payment_value")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
52107,03caa2c082116e1d31e67e9ae3700499,1,credit_card,1,13664.08
34370,736e1922ae60d0d6a89247b851902527,1,boleto,1,7274.88
41419,0812eb902a67711a1cb742b3cdaa65ae,1,credit_card,8,6929.31
49581,fefacc66af859508bf1a7934eab1e97f,1,boleto,1,6922.21
85539,f5136e38d1a14a4dbd87dff67da82701,1,boleto,1,6726.66
62409,2cc9089445046817a7539d90805e6e5a,1,boleto,1,6081.54
43232,a96610ab360d42a2e5335a3998b4718a,1,credit_card,10,4950.34
70320,b4c4b76c642808cbe472a32b86cddc95,1,credit_card,5,4809.44
6440,199af31afc78c699f0dbf71fb178d4d4,1,credit_card,8,4764.34
67546,8dbc85d1447242f3b127dda390d56e19,1,credit_card,8,4681.78


In [32]:
payments.nlargest(10, "payment_installments")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
2970,70b7e94ea46d3e8b5bc12a50186edaf0,1,credit_card,24,274.84
10791,859f516f2fc3f95772e63c5757ab0d5b,1,credit_card,24,609.56
12307,ff36cbc44b8f228e0449c92ef089c843,1,credit_card,24,756.49
18512,2b7dbe9be72b8f9733844c31055c0825,1,credit_card,24,345.39
21713,6ae2e8b8fac02522481d2a2f4ca4412c,1,credit_card,24,433.43
23024,90f864fe19d11549fa01eb81c4dd87e3,1,credit_card,24,588.58
36088,84d2098c97827c6327ed4d7be95e1fc8,1,credit_card,24,286.78
50401,ffb18bf111fa70edf316eb0390427986,1,credit_card,24,617.24
52846,63dbe0c8e63e5f1b4deec09d4f044a7f,1,credit_card,24,771.69
55094,fcbb6af360b31b05460c2c8e524588c0,1,credit_card,24,1194.38


In [33]:
payments.nlargest(10, "payment_sequential")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
39108,fa65dad1b0e818e3ccc5cb0e39231352,29,voucher,1,19.26
39111,fa65dad1b0e818e3ccc5cb0e39231352,28,voucher,1,29.05
4885,fa65dad1b0e818e3ccc5cb0e39231352,27,voucher,1,66.02
32393,ccf804e764ed5650cd8759557269dc13,26,voucher,1,23.10
79587,fa65dad1b0e818e3ccc5cb0e39231352,26,voucher,1,28.27
24879,fa65dad1b0e818e3ccc5cb0e39231352,25,voucher,1,3.68
39132,ccf804e764ed5650cd8759557269dc13,25,voucher,1,1.53
51816,ccf804e764ed5650cd8759557269dc13,24,voucher,1,2.79
99213,fa65dad1b0e818e3ccc5cb0e39231352,24,voucher,1,0.42
60241,ccf804e764ed5650cd8759557269dc13,23,voucher,1,1.03


## 4.Decision



No missing values were found.  
No duplicate rows were found.  
Data types are appropriate for the columns.  
Three not_defined payment types were identified and retained because the actual payment type cannot be determined.  
Two records with payment_installments = 0 were identified and retained because the correct value cannot be determined from the available data.  
Zero payment values were investigated and retained.  
High installment and payment sequential values were investigated and considered valid.  
Payment value outliers were retained because large values do not necessarily indicate data errors.  